# Data splitting: lab 08 extension assignments

This notebook contains short executable examples, assignments and hints. Write your own procedure in the empty code cells beneath each step. Examples use separate miniature datasets and variables prefixed with `demo_`. Tasks A, B and C are independent: each uses its own data file. All files contain synthetic data.

Run the notebook from `08-Data-preprocessing` or the repository root. The working directory must allow access to the supplied `data` folder. Dataset descriptions are in `data/README.md`; packages are listed in the root `requirements.txt`.

## Setup

The next cell imports tools for both the examples and your own work. After these imports, tasks A, B and C are independent.

When loading your first task, store the data-directory path in `DATA` as a `Path` object. From the notebook directory it is `data`; from the repository root it is `08-Data-preprocessing/data`. `pd.read_csv` accepts a file path. `display` shows tables, `np` refers to NumPy, and `pd` to pandas.

In [ ]:
from pathlib import Path  # File paths for your own tasks.
from io import StringIO  # A small CSV example held in memory.
import numpy as np  # Numeric arrays and missing values.
import pandas as pd  # Tables, CSV files and timestamps.
from IPython.display import display  # Readable table display.
from sklearn.model_selection import train_test_split, GroupShuffleSplit, TimeSeriesSplit  # Three data-splitting methods.
from sklearn.tree import DecisionTreeClassifier  # A simple classification model.
from sklearn.pipeline import Pipeline  # Fit preprocessing together with the model.
from sklearn.impute import SimpleImputer  # Fill missing inputs using training data.
from sklearn.linear_model import Ridge  # A regression model for a numeric target.
from sklearn.metrics import accuracy_score, mean_absolute_error  # Classification and regression metrics.

### Example: loading a table

Three example rows are held in memory. To read a file, `read_csv` accepts a path instead of `StringIO`, for example a `Path` object. This example does not use the assignment data.

In [ ]:
demo_csv = StringIO("category,value\nA,2\nB,5\nA,7\n")  # The contents of a separate miniature CSV.
demo_table = pd.read_csv(demo_csv)  # The same loading API used for files.
display(demo_table)  # Show the loaded columns and rows.
demo_values = demo_table["value"].to_numpy()  # Convert one column to an array.
print(demo_values)  # An array can then be selected by index.

## A · Categories and intervals

**Goal:** compare stratification by a category with stratification by intervals of a numeric variable. No model is fitted yet.

Load `data/stratification-records.csv` into `a_table`. It contains 120 rows, a `state` column with labels `normal`, `warning`, `fault`, and a numeric `load` column. There are no missing values or group/time dependencies. Row order has no temporal meaning.

Create a category array `a_state`, numeric array `a_load`, and row-position array `a_indices`. A column can be converted with `.to_numpy()`; positions can be created with `numpy.arange`. Inspect category counts. Both following splits must use 25% of rows for testing and `random_state=42`.

### Example: splitting by category

The miniature dataset has two categories. The split operates on row positions; the same positions then select the corresponding labels.

In [ ]:
demo_labels = np.array(["A"] * 8 + ["B"] * 4)  # Two categories with different counts.
demo_positions = np.arange(len(demo_labels))  # Positions of the rows to split.
demo_train, demo_test = train_test_split(demo_positions, test_size=0.5, random_state=0, stratify=demo_labels)  # Split while preserving category proportions.
print(demo_train, demo_test)  # Separate training and test positions.
print(np.unique(demo_labels[demo_test], return_counts=True))  # Categories and their counts in the example test.

### A1 · Categories

Use `train_test_split` to divide `a_indices` into `a_train_idx` and `a_test_idx`, preserving the proportions in `a_state`. Build a table of each category’s count in the full data, training and test sets. Verify that the partitions do not overlap and together contain every row.

<details><summary>Hint</summary>

`test_size` controls test size, `random_state` reproducibility, and `stratify` the labels used for stratification. The last argument requires one label per row. Returned arrays are ordered training, test. `value_counts` supplies counts; sets of indices can establish disjointness.

</details>

### Example: quantile intervals

For stratification, a numeric variable is converted into interval labels. The two intervals in this small example contain equal numbers of values but have different widths.

In [ ]:
demo_numbers = np.array([2, 3, 4, 6, 9, 12, 17, 30])  # A separate set of eight distinct values.
demo_bins = pd.qcut(demo_numbers, q=2, labels=False)  # Labels of two quantile intervals.
print(demo_bins)  # Each original value has one interval label.
demo_bin_train, demo_bin_test = train_test_split(np.arange(len(demo_numbers)), test_size=0.5, random_state=0, stratify=demo_bins)  # Interval labels determine the stratification.
print(np.unique(demo_bins[demo_bin_test], return_counts=True))  # Interval representation in the example test.

### A2 · Intervals

Create `a_bins`: four quantile intervals of `a_load`, labelled with integers. Use these labels to split the same row positions again into `a_interval_train_idx` and `a_interval_test_idx`. Build a table of interval counts in the full, training and test data. Also inspect the original category proportions in this test set.

<details><summary>Hint</summary>

For `pandas.qcut`, `q` controls the number of intervals and `labels=False` returns numeric labels. These new labels can become the `stratify` argument. Test counts cannot be exact quarters when the requested count is not an integer.

</details>

Here intervals form part of the sampling design for independent observations. If they were model features, their boundaries would be fitted on training data only.

**Output:** tables of category and interval counts.

**For reflection:** What did each split preserve? Why does quantile stratification alone not guarantee state proportions? What problem can combined “state × interval” categories cause?

[Split documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [quantile intervals](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html).

## B · New devices

**Goal:** estimate model quality for devices absent from training.

Load `data/grouped-records.csv` into `b_data`. It contains 36 devices with eight measurements each. Select only `temperature` and `vibration` into `b_features`, target `state` into `b_target`, and identifier `group_id` into `b_groups`. The identifier is not a model input. No values are missing.

Inspect row and device counts. Measurements of the same device are related, so they must stay in one partition when evaluating predictions for new devices.

### Example: keeping whole groups together

Four groups each contain two rows. `test_size` refers to the number of groups; every row from a group stays in the same partition.

In [ ]:
demo_groups = np.repeat(["G1", "G2", "G3", "G4"], 2)  # Identifiers for four small groups.
demo_features = np.arange(8).reshape(-1, 1)  # Eight rows with one illustrative input.
demo_group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)  # One split of entire groups.
demo_group_train, demo_group_test = next(demo_group_splitter.split(demo_features, groups=demo_groups))  # Pass groups to the split method.
print(np.unique(demo_groups[demo_group_train]))  # Groups assigned to training.
print(np.unique(demo_groups[demo_group_test]))  # Different groups assigned to test.

### B1 · Group split

Create `GroupShuffleSplit` with `n_splits=1`, `test_size=0.25`, and `random_state=42`. Obtain position arrays `b_train_idx` and `b_test_idx`, actually passing groups to the split. Report device and row counts in both partitions, check that device sets are disjoint, and compare class proportions.

<details><summary>Hint</summary>

Settings belong in the constructor. `.split` receives features, targets and `groups`, yielding pairs of indices. `next` obtains the first pair. Here `test_size` specifies a proportion of devices. `.iloc` selects rows by position; `.nunique()` counts distinct identifiers.

</details>

### Example: fitting and predicting a class

Training and test rows are given separately here. Prediction receives only inputs; actual targets are used later to compute accuracy.

In [ ]:
demo_class_inputs = np.array([[1], [2], [4], [5]])  # Four training samples with one input.
demo_class_targets = np.array([0, 0, 1, 1])  # Training classes in matching order.
demo_classifier = DecisionTreeClassifier(max_depth=1, random_state=0)  # A small tree with fixed settings.
demo_classifier.fit(demo_class_inputs, demo_class_targets)  # Fit using training rows only.
demo_class_prediction = demo_classifier.predict([[1.5], [4.5]])  # Two new inputs without their target classes.
print(demo_class_prediction, accuracy_score([0, 1], demo_class_prediction))  # Predicted classes and their accuracy.

### B2 · Model and interpretation

Fit `DecisionTreeClassifier(max_depth=3, random_state=42)` using training devices only. Predict the test rows and compute `accuracy_score`. Settings are fixed in advance and must not change in response to test results.

<details><summary>Hint</summary>

`.fit` receives training features and targets; `.predict` receives test features only. `accuracy_score` compares actual test targets with predictions. Device identifiers are used only for group splitting.

</details>

**Output:** device counts, class proportions, and accuracy.

**For reflection:** What different task would a random split of individual measurements evaluate? Why might group splitting alone fail to preserve class proportions?

[Group split documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html).

## C · Next-hour prediction

**Goal:** compare expanding and sliding training windows, both with a one-hour gap and with no gap.

Load `data/time-series.csv` into `c_data`; parse `timestamp` and `target_available_time` as times with `parse_dates`. Verify unique timestamps in chronological order. Observations have a regular hourly frequency.

At `timestamp`, inputs `current_value` and `lag1` are known. Target `next_value` becomes known at `target_available_time`, one hour later. Equality with the fitting time means the newly arrived measurement is already available.

Store the first 80% of rows in `c_development`. Reserve the remaining 20% in `c_holdout` as a final test set that is not evaluated in this task. Create `c_features` with the two inputs and `c_target` with the target from the development portion. In a working copy of the inputs, replace every thirteenth `lag1`, starting with the first, by a missing value for this experiment; leave the CSV file unchanged.

<details><summary>Hint</summary>

Calculate the boundary from the number of rows; `.iloc` selects consecutive portions. `.copy()` makes an independent working table. `numpy.nan` represents a missing numeric value. `.is_monotonic_increasing` checks time order and `.is_unique` checks uniqueness.

</details>

### Example: a gap between time windows

Twelve hourly rows are enough for two small validation windows. The example limits training to four rows. Without `max_train_size`, the history would expand.

In [ ]:
demo_times = pd.date_range("2025-02-01", periods=12, freq="h")  # Timestamps of a separate example series.
for demo_gap in [0, 1]:  # Compare no gap against one omitted row.
    demo_time_splitter = TimeSeriesSplit(n_splits=2, test_size=2, gap=demo_gap, max_train_size=4)  # Two validation windows of two hours each.
    for demo_time_train, demo_time_valid in demo_time_splitter.split(demo_times):  # Indices for each temporal round.
        print(demo_gap, demo_time_train, demo_time_valid)  # Show the gap and training-history length.
        print(demo_times[demo_time_train[-1]], demo_times[demo_time_valid[0]])  # Last training time and first validation time.

### C1 · Time windows

Create four combinations: expanding and sliding windows, each for `gap=1` and `gap=0`. Use `TimeSeriesSplit` with `n_splits=4` and `test_size=24`. Sliding windows additionally limit training to at most 72 rows through `max_train_size`.

For every combination, obtain training and validation indices within `c_features` only. Display counts and times for each round. Check that training precedes validation and the final test remains outside the experiment.

<details><summary>Hint</summary>

`.split` yields index pairs that can be iterated over or converted to a list. Without a `max_train_size` limit, all history is retained. `gap` omits rows between training and validation; it neither replaces target-availability checks nor guarantees independence.

</details>

### Example: when a target is known

Three candidate targets arrive at different times. Equality with fitting time means the newly arrived target can already be used here.

In [ ]:
demo_available = pd.to_datetime(["2025-02-01 09:00", "2025-02-01 10:00", "2025-02-01 11:00"])  # Times when candidate targets become known.
demo_fit_time = pd.Timestamp("2025-02-01 10:00")  # A fixed fitting time.
demo_known = demo_available <= demo_fit_time  # The available-target mask includes equality.
print(demo_known, demo_available[demo_known])  # The last target is not yet available.

### Example: a pipeline and prediction error

Missing-value replacement is fitted together with the model using the small training set only. MAE (mean absolute error) compares predictions with the outcomes of separate validation rows.

In [ ]:
demo_reg_inputs = np.array([[1.0], [np.nan], [3.0], [4.0]])  # Four training rows with one missing value.
demo_reg_targets = np.array([2.0, 3.0, 4.0, 5.0])  # Known training targets.
demo_reg_model = Pipeline([("impute", SimpleImputer(strategy="median")), ("model", Ridge(alpha=0.5))])  # Imputation and model form one procedure.
demo_reg_model.fit(demo_reg_inputs, demo_reg_targets)  # Both imputation and model are fitted on these rows only.
demo_reg_prediction = demo_reg_model.predict([[2.0], [5.0]])  # Predictions for two separate validation inputs.
demo_reg_observed = np.array([3.0, 6.0])  # Actual validation outcomes for scoring.
print(demo_reg_prediction)  # The calculated numeric predictions.
print(mean_absolute_error(demo_reg_observed, demo_reg_prediction))  # Mean absolute error in the same units as the target.

### C2 · Available targets and model

For each round, `c_fit_time` is the first validation `timestamp`. Retain only candidate training indices whose `target_available_time` is no later than `c_fit_time`. Report the retained row count and latest available-target time.

Create a fresh `Pipeline` with `SimpleImputer(strategy="median")` and `Ridge(alpha=1.0)` in each round. Fit the entire pipeline on permitted rows only and predict validation inputs. Compare it with a persistence baseline: the next value equals the currently observed `current_value`. Compute the mean absolute error (MAE) for both methods using `mean_absolute_error`.

The model is fitted at the start of each validation block and stays fixed within that block. Each subsequent hourly forecast uses the measurement available at that hour. This is not a whole-day forecast from a single origin. Preprocessing must not use future values.

<details><summary>Hint: availability</summary>

Compare availability times only for candidate training rows. The Boolean mask must have the same length as their index array; `.to_numpy()` converts it to an array. A training row’s timestamp alone is insufficient because its target arrives later.

</details>

<details><summary>Hint: pipeline and metrics</summary>

A pipeline receives named steps. `.fit` receives training inputs and targets; `.predict` receives validation inputs. The median is therefore fitted within that training window. MAE compares actual validation targets with predictions in matching order. `c_holdout` belongs in neither error calculation nor method selection.

</details>

### Task C output

Create a table of all 16 validation rounds.

**For reflection:**

- In which rounds did the model outperform the persistence baseline? Does the conclusion change with the window or gap?
- Why is the median fitted again in every round?
- Did the zero gap also satisfy target availability? What does equality of availability time and fitting time mean?
- What would change when forecasting a whole day from one initial measurement?

The final test remains reserved. These are validation results, not a final test score.

[Time windows and gap](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), [pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [imputation](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html).